In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F

# model_name = "allegro/herbert-large-cased"
# model_name = "dkleczek/bert-base-polish-cased-v1"
model_name = "nlpaueb/legal-bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to("xpu")

max_token_length = 512

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Exception ignored in: <function tqdm.__del__ at 0x00000285F45820C0>
Traceback (most recent call last):
  File "c:\Users\jasie\Desktop\Polish-Legal-Synonyms\existing-solutions-analysis\bert\env\Lib\site-packages\tqdm\std.py", line 1148, in __del__
    self.close()
  File "c:\Users\jasie\Desktop\Polish-Legal-Synonyms\existing-solutions-analysis\bert\env\Lib\site-packages\tqdm\notebook.py", line 279, in close
    self.disp(bar_style='danger', check_delay=False)
AttributeError: 'tqdm' object has no attribute 'disp'


In [2]:
def get_embedding(text):
    tokens = tokenizer(text, return_tensors="pt", return_offsets_mapping=True, add_special_tokens=True).to("xpu")
    offsets = tokens.pop("offset_mapping")[0].cpu().tolist()

    with torch.no_grad():
        outputs = model(**tokens)
    embeddings = outputs.last_hidden_state.squeeze(0)

    input_ids = tokens["input_ids"][0].cpu().tolist()
    subtokens = tokenizer.convert_ids_to_tokens(input_ids)

    word_embeddings = []
    words = []

    current_word = ""
    current_vecs = []
    last_end = None

    for subtoken, (start, end), vec in zip(subtokens, offsets, embeddings):
        if start == 0 and end == 0:
            continue

        piece = text[start:end]

        if last_end is None or start == last_end:  
            # kontynuacja tego samego słowa
            current_word += piece
            current_vecs.append(vec)
        else:
            # zapisujemy poprzednie słowo
            if current_vecs:
                word_embeddings.append(torch.mean(torch.stack(current_vecs), dim=0))
                words.append(current_word)

            # zaczynamy nowe słowo
            current_word = piece
            current_vecs = [vec]

        last_end = end

    # dodaj ostatnie słowo
    if current_vecs:
        word_embeddings.append(torch.mean(torch.stack(current_vecs), dim=0))
        words.append(current_word)

    return word_embeddings, words

In [3]:
with open("tekst.txt", "r", encoding="utf-8") as f:
    text = f.read()

seen = set()
unique_words = []
for word in text.split():
    word = word.strip('.,!?;"()[]{}<>«»„”\'`')
    if word.lower() not in seen:
        seen.add(word.lower())
        unique_words.append(word)
text = " ".join(unique_words)

# podziel tekst na kawałki <= max_token_length
def chunk_text(text, max_tokens):
    words = text.split()
    chunks = []
    current_chunk = ""

    for word in words:
        test_chunk = current_chunk + " " + word if current_chunk else word
        tokenized = tokenizer(test_chunk, return_tensors="pt")
        if tokenized.input_ids.size(1) <= max_tokens:
            current_chunk = test_chunk
        else:
            if current_chunk:
                chunks.append(current_chunk)
            current_chunk = word

    if current_chunk:
        chunks.append(current_chunk)

    return chunks

chunks = chunk_text(text, max_token_length - 2)  # -2 na [CLS] i [SEP]

Token indices sequence length is longer than the specified maximum sequence length for this model (513 > 512). Running this sequence through the model will result in indexing errors


In [4]:
# #maciez podobienstwa
#nie działa dla większych tekstów

# embedding = []
# words = []

# for chunk in chunks:
#     emb, wds = get_embedding(chunk)
#     embedding.extend(emb)
#     words.extend(wds)

# valid_indices = [i for i, w in enumerate(words) if not w.startswith("##") and w not in ["[CLS]", "[SEP]"]]
# filtered_words = [words[i] for i in valid_indices]
# filtered_embeddings = torch.stack([embedding[i] for i in valid_indices])

# similarity_matrix = F.cosine_similarity(
#     filtered_embeddings.unsqueeze(1),  # [n,1,d]
#     filtered_embeddings.unsqueeze(0),  # [1,n,d]
#     dim=-1
# )

# threshold = 0.85
# visited = set()
# groups = []

# for i in range(len(filtered_words)):
#     if i in visited:
#         continue
#     group = [filtered_words[i]]
#     visited.add(i)
#     for j in range(len(filtered_words)):
#         if j not in visited and similarity_matrix[i, j] > threshold and i != j:
#             group.append(filtered_words[j])
#             visited.add(j)
#     if len(group) > 1:
#         groups.append(group)


# for idx, g in enumerate(groups, 1):
#     print(f"Grupa {idx}: {g}")

In [5]:
# kmeans
from sklearn.cluster import KMeans

embedding = []
words = []

for chunk in chunks:
    emb, wds = get_embedding(chunk)
    embedding.extend(emb)
    words.extend(wds)

valid_indices = [i for i, w in enumerate(words) if not w.startswith("##") and w not in ["[CLS]", "[SEP]"]]
filtered_words = [words[i] for i in valid_indices]
filtered_embeddings = torch.stack([embedding[i] for i in valid_indices]).cpu().numpy()

print(len(filtered_words), filtered_embeddings.shape)

k = 300
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
labels = kmeans.fit_predict(filtered_embeddings)

clusters = {}
for word, label in zip(filtered_words, labels):
    clusters.setdefault(label, []).append(word)

for label, group in clusters.items():
    print(f"Grupa {label}: {group}")

1376 (1376, 768)
Grupa 129: ['po', 'na']
Grupa 249: ['rozpoznaniu', 'rozprawie']
Grupa 204: ['w']
Grupa 110: ['dniu', 'dnia']
Grupa 96: ['20', '15', '19']
Grupa 166: ['maja', 'lutego']
Grupa 73: ['2015', '2010']
Grupa 270: ['r']
Grupa 50: ['Gdańsku', 'warunki', 'zakreślone', 'pisemnym', 'zobowiązaniem', 'oznaczonej', 'prawnym', 'polega', 'zobowiązują']
Grupa 211: ['sprawy', 'przyznanej', 'wstępnej', 'Zależność']
Grupa 251: ['z']
Grupa 53: ['powództwa', 'Powód', 'powództwo', 'Powyższe', 'powódce']
Grupa 225: ['spółki', 'Spółce', 'właściwe', 'zł', 'wywodził', 'zobowiązał', 'zapłacić', 'powstałe', 'Wydział', 'uwzględnił', 'złożył', 'wniósł', 'właściwy', 'podniósł', 'protokołu', 'wystawił', 'dłużej']
Grupa 11: ['ograniczoną', 'sądowych', 'faktyczny:', 'nazwą', 'dokumentów', 'dokumentacja', 'projektową', 'wyznaczając', 'dotycząca', 'Tymczasem', 'Faktem']
Grupa 262: ['odpowiedzialnością', 'odpowiedzialności', 'odpowiedzialność']
Grupa 58: ['J', 'S', 'wad', 'a', 'stron']
Grupa 20: ['przeciwko

In [11]:
#faiss
import faiss

embedding = []
words = []

for chunk in chunks:
    emb, wds = get_embedding(chunk)
    embedding.extend(emb)
    words.extend(wds)

valid_indices = [i for i, w in enumerate(words)
                 if not w.startswith("##") and w not in ["[CLS]", "[SEP]"]]
filtered_words = [words[i] for i in valid_indices]
filtered_embeddings = torch.stack([embedding[i] for i in valid_indices])

emb_norm = F.normalize(filtered_embeddings, p=2, dim=1).cpu().numpy().astype("float32")

index = faiss.IndexFlatIP(emb_norm.shape[1])
index.add(emb_norm)

k = 10  # top k najbardziej podobnych słów
S, I = index.search(emb_norm, k)  # S = podobieństwa, I = indeksy

threshold = 0.85
visited = set()
groups = []

for i in range(len(filtered_words)):
    if i in visited:
        continue
    group = {filtered_words[i]}
    visited.add(i)
    for j, sim in zip(I[i][1:], S[i][1:]):  # pomijamy [0], bo to to samo słowo
        if sim > threshold and j not in visited:
            group.add(filtered_words[j])
            visited.add(j)
    if len(group) > 1:
        groups.append(list(group))

for idx, g in enumerate(groups, 1):
    print(f"Grupa {idx}: {g}")


Grupa 1: ['maja', 'lutego']
Grupa 2: ['Powód', 'powództwo', 'Powyższe', 'powództwa']
Grupa 3: ['oddziałuje', 'podniósł', 'wniósł', 'powstałe', 'Spółce', 'wywodził', 'uwzględnił', 'Wydział', 'reguł', 'spółki']
Grupa 4: ['odpowiedzialności', 'odpowiedzialnością']
Grupa 5: ['przeciwko', 'nieistotne', 'Akcyjnej']
Grupa 6: ['nieodwołalnie', 'Załączona', 'dług', 'zapłaty', 'element', 'zapłatę', 'dotyczyła', 'tytułem', 'pełni', 'załączenia']
Grupa 7: ['pozwanego', 'pozwanej', 'Pozwany']
Grupa 8: ['II/', 'I/']
Grupa 9: ['zasądza', 'zasadą', 'zasądzenie', 'uwzględnienie']
Grupa 10: ['94.000', '2.700']
Grupa 11: ['uprawniały', 'był', 'zgłoszenie', 'złotych', 'pełnomocnictwa', 'złożono', 'nieprawidłowe', 'prawidłowy', 'potwierdził', 'według']
Grupa 12: ['kosztów', 'zwrotu']
Grupa 13: ['zasądzenia', 'zastępstwa', 'procesowego']
Grupa 14: ['postępowania', 'postępowanie', 'apelacyjne']
Grupa 15: ['zapłacić', 'złożył', 'zobowiązał', 'przedłożenia', 'protokołu', 'właściwe', 'właściwy']
Grupa 16: ['wła